Feature engineering

En este notebook se transforman las reviews limpias en variables numéricas para poder entrenar modelos de Machine Learning en la siguiente fase.

In [1]:
import pandas as pd

In [2]:
from google.colab import files

uploaded = files.upload()

Saving amazon_reviews_clean.csv to amazon_reviews_clean.csv


In [5]:
df = pd.read_csv("amazon_reviews_clean.csv")

df.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase', 'review',
       'review_length', 'review_clean', 'sentiment'],
      dtype='object')

In [6]:
df[["rating", "sentiment", "review_clean"]].head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             50000 non-null  float64
 1   title              49995 non-null  object 
 2   text               49998 non-null  object 
 3   images             50000 non-null  object 
 4   asin               50000 non-null  object 
 5   parent_asin        50000 non-null  object 
 6   user_id            50000 non-null  object 
 7   timestamp          50000 non-null  int64  
 8   helpful_vote       50000 non-null  int64  
 9   verified_purchase  50000 non-null  bool   
 10  review             50000 non-null  object 
 11  review_length      50000 non-null  int64  
 12  review_clean       49945 non-null  object 
 13  sentiment          50000 non-null  object 
dtypes: bool(1), float64(1), int64(3), object(9)
memory usage: 5.0+ MB


In [7]:
#eliminar los nulos
df = df.dropna(subset=["review_clean"])
df["review_clean"].isnull().sum()

np.int64(0)

In [9]:
# Variables de entrada y salida

X = df["review_clean"]
y = df["sentiment"]

In [11]:
print("Número de reviews:", len(X))
print("Número de etiquetas:", len(y))

print("\nDistribución del sentimiento:")
print(y.value_counts())

Número de reviews: 49945
Número de etiquetas: 49945

Distribución del sentimiento:
sentiment
Positivo    40277
Negativo     6259
Neutro       3409
Name: count, dtype: int64


**División del conjunto de datos**

El conjunto de datos se divide en un 80 % para entrenamiento y un 20 % para prueba.

Además, se utiliza el parámetro `stratify=y` para mantener la misma proporción de opiniones positivas, negativas y neutras en ambos conjuntos, evitando que el desbalance de clases afecte a la evaluación del modelo.

In [12]:
print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

print("\nDistribución Train:")
print(y_train.value_counts())

print("\nDistribución Test:")
print(y_test.value_counts())

Tamaño entrenamiento: (39956,)
Tamaño prueba: (9989,)

Distribución Train:
sentiment
Positivo    32222
Negativo     5007
Neutro       2727
Name: count, dtype: int64

Distribución Test:
sentiment
Positivo    8055
Negativo    1252
Neutro       682
Name: count, dtype: int64


In [10]:
#train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Vectorización TF-IDF**

Los modelos de Machine Learning no pueden trabajar directamente con texto.  
Por ello, se utiliza TF-IDF para transformar las reviews limpias en variables numéricas.

TF-IDF da más importancia a las palabras que son relevantes dentro de una review, pero que no aparecen constantemente en todos los textos.

In [13]:
#vectorizacion tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer(
    max_features=5000
)

X_train_tfidf = vectorizador.fit_transform(X_train)
X_test_tfidf = vectorizador.transform(X_test)

In [14]:
#ver tamaño de las matrices
print("Train:", X_train_tfidf.shape)
print("Test:", X_test_tfidf.shape)

Train: (39956, 5000)
Test: (9989, 5000)


In [17]:
#vocabulario aprendido
print(vectorizador.get_feature_names_out()[:20])
print(vectorizador.get_feature_names_out()[-20:])

['aa' 'aaa' 'ab' 'ability' 'able' 'absolute' 'absolutely' 'abuse' 'ac'
 'accent' 'accept' 'acceptable' 'accepted' 'accepts' 'access' 'accessed'
 'accessible' 'accessing' 'accessoriesbr' 'accessory']
['yetbr' 'yo' 'yoga' 'youbr' 'youd' 'youll' 'young' 'younger' 'youre'
 'youtube' 'youve' 'yr' 'zero' 'zip' 'zipper' 'zippered' 'zone' 'zoom'
 'zooming' 'zune']


In [18]:
import pickle

with open("X_train_tfidf.pkl", "wb") as archivo:
    pickle.dump(X_train_tfidf, archivo)

with open("X_test_tfidf.pkl", "wb") as archivo:
    pickle.dump(X_test_tfidf, archivo)

with open("y_train.pkl", "wb") as archivo:
    pickle.dump(y_train, archivo)

with open("y_test.pkl", "wb") as archivo:
    pickle.dump(y_test, archivo)

with open("vectorizador_tfidf.pkl", "wb") as archivo:
    pickle.dump(vectorizador, archivo)

In [19]:
from google.colab import files

files.download("X_train_tfidf.pkl")
files.download("X_test_tfidf.pkl")
files.download("y_train.pkl")
files.download("y_test.pkl")
files.download("vectorizador_tfidf.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Conclusiones**

En este notebook se ha realizado la fase de Feature Engineering.

Se ha dividido el dataset en entrenamiento y prueba manteniendo la proporción de clases mediante `stratify=y`. Posteriormente, se ha aplicado TF-IDF para transformar las reviews limpias en variables numéricas.

El resultado final son matrices de entrenamiento y prueba con 5.000 características, listas para ser utilizadas en el entrenamiento de modelos de Machine Learning.